In [ ]:
# Code for generating 10000 proxy

import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import skew, kurtosis

# Assume the data file is named "data.nc". Replace it if your filename is different.
data_file = 'IFS_t2m_hottest_month_detrended_1981_2023.nc'

# Load the dataset
dataset = xr.open_dataset(data_file)

# Define time range and regions
time_range = pd.to_datetime(['1981', '2023'])
regions = dataset.region_id.values

# Initialize arrays to store results
num_samples = 10000
output_means = np.zeros((num_samples, len(regions)))
output_stds = np.zeros((num_samples, len(regions)))
output_skews = np.zeros((num_samples, len(regions)))
output_kurts = np.zeros((num_samples, len(regions)))

# Loop over each region
for i, region in enumerate(regions):
    print(f"Processing region {region} ({i + 1}/{len(regions)})")

    # Extract time series for this region during 1981–2023
    region_data = dataset.sel(region_id=region, year=slice('1981', '2023')).t2m_hottest_month_detrended
        
    for sample_idx in range(num_samples):
        # Randomly sample with replacement from 25 members to build a sequence ordered by year
        sampled_series = []
        for year_idx in range(43):  # Loop over each year
            number_idx = np.random.choice(25)
                
            # Extract the sequence for this year
            series = region_data[number_idx, year_idx]
            sampled_series.append(series)

        # Convert to NumPy array (length = 43)
        sampled_series = np.array(sampled_series)

        # Compute statistics
        output_means[sample_idx, i] = np.mean(sampled_series)
        output_stds[sample_idx, i] = np.std(sampled_series)
        output_skews[sample_idx, i] = skew(sampled_series)
        output_kurts[sample_idx, i] = kurtosis(sampled_series)

# Save results to a new NetCDF file
output_dataset = xr.Dataset(
    {
        'means': (['sample', 'region_id'], output_means),
        'stds': (['sample', 'region_id'], output_stds),
        'skews': (['sample', 'region_id'], output_skews),
        'kurts': (['sample', 'region_id'], output_kurts),
    },
    coords={
        'sample': np.arange(num_samples),
        'region_id': regions,
    }
)

# Save dataset to file
output_file = 'IFS_proxy_statistics_hottest_regional_1981_2024_land.nc'
output_dataset.to_netcdf(output_file)

print(f"Computation complete, results saved to {output_file}")


In [ ]:
# code for using the 10000 proxy and ERA5 to test the region medel fidelity
import numpy as np
import xarray as xr
import scipy.stats as stats
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from scipy.signal import detrend
from scipy.stats import linregress

def load_and_test(era5_file, proxy_file, region_grid_ds):
    era5_dataset = xr.open_dataset(era5_file)
    proxy_dataset = xr.open_dataset(proxy_file)
    regions = era5_dataset.region_id.values

    era5_means = np.zeros(len(regions))
    era5_stds = np.zeros(len(regions))
    era5_skews = np.zeros(len(regions))
    era5_kurts = np.zeros(len(regions))

    proxy_means = proxy_dataset.means.values
    proxy_stds = proxy_dataset.stds.values
    proxy_skews = proxy_dataset.skews.values
    proxy_kurts = proxy_dataset.kurts.values

    pass_test = np.zeros(len(regions))

    for i, region in enumerate(regions):
        era5_data = era5_dataset.sel(region_id=region, year=slice(1981, 2024)).t2m_hottest
        era5_data_detrended = detrend(era5_data, axis=0)

        era5_means[i] = era5_data_detrended.mean()
        era5_stds[i] = era5_data_detrended.std()
        era5_skews[i] = stats.skew(era5_data_detrended)
        era5_kurts[i] = stats.kurtosis(era5_data_detrended)

        mean_5th, mean_95th = np.percentile(proxy_means[:, i], [5, 95])
        std_5th, std_95th = np.percentile(proxy_stds[:, i], [5, 95])
        skew_5th, skew_95th = np.percentile(proxy_skews[:, i], [5, 95])
        kurt_5th, kurt_95th = np.percentile(proxy_kurts[:, i], [5, 95])

        mean_pass = mean_5th <= era5_means[i] <= mean_95th
        std_pass = std_5th <= era5_stds[i] <= std_95th
        skew_pass = skew_5th <= era5_skews[i] <= skew_95th
        kurt_pass = kurt_5th <= era5_kurts[i] <= kurt_95th

        if std_pass and skew_pass and kurt_pass:
            pass_test[i] = 0.5
        if std_pass and skew_pass and kurt_pass and mean_pass:
            pass_test[i] = 1

    region_grid = region_grid_ds.region_id
    pass_test_grid = xr.full_like(region_grid, np.nan, dtype=float)

    for i, rid in enumerate(regions):
        pass_test_grid = pass_test_grid.where(region_grid != rid, pass_test[i])

    return pass_test_grid

def load_and_test_trend(era5_file, proxy_file, region_grid_file):
    era5_dataset = xr.open_dataset(era5_file)
    region_grid_ds = xr.open_dataset(region_grid_file)
    proxy_dataset = xr.open_dataset(proxy_file)

    regions = era5_dataset.region_id.values
    years = era5_dataset.sel(year=slice(1981, 2024)).year

    era5_data_raw = era5_dataset.sel(year=slice(1981, 2024)).t2m_hottest

    def calc_slope_intercept(y, x):
        mask = np.isfinite(y)
        if mask.sum() < 2:
            return np.nan, np.nan
        slope, intercept, _, _, _ = linregress(x[mask], y[mask])
        return slope, intercept

    slope, intercept = xr.apply_ufunc(
        calc_slope_intercept,
        era5_data_raw,
        years,
        input_core_dims=[['year'], ['year']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[era5_data_raw.dtype, era5_data_raw.dtype]
    )

    proxy_slope = proxy_dataset.slope
    proxy_intercept = proxy_dataset.intercept

    slope_p05 = proxy_slope.quantile(0.05, dim='sample')
    slope_p95 = proxy_slope.quantile(0.95, dim='sample')
    intercept_p05 = proxy_intercept.quantile(0.05, dim='sample')
    intercept_p95 = proxy_intercept.quantile(0.95, dim='sample')

    era5_region_ids = era5_dataset.region_id.values
    proxy_region_ids = proxy_dataset.region_id.values

    region_id_to_index = {rid: i for i, rid in enumerate(proxy_region_ids)}
    proxy_idx = [region_id_to_index[r] for r in era5_region_ids]

    slope_05 = slope_p05.isel(region_id=proxy_idx).values
    slope_95 = slope_p95.isel(region_id=proxy_idx).values
    intercept_05 = intercept_p05.isel(region_id=proxy_idx).values
    intercept_95 = intercept_p95.isel(region_id=proxy_idx).values

    era5_slope = slope.values
    era5_intercept = intercept.values

    pass_slope = (era5_slope >= slope_05) & (era5_slope <= slope_95)
    pass_intercept = (era5_intercept >= intercept_05) & (era5_intercept <= intercept_95)

    category = np.where(pass_slope & pass_intercept, 1,
                        np.where(pass_slope, 0.5, 0))

    region_mask = region_grid_ds['region_id']
    region_plot = np.full(region_mask.shape, np.nan)
    for i, region_id in enumerate(era5_region_ids):
        region_plot[region_mask.values == region_id] = category[i]

    return region_plot

# Load region_grid
region_grid_ds = xr.open_dataset("/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/region_mask_1deg.nc")

# Load IFS and DP3 test results
grid_ifs = load_and_test(
    "Berkeley_hottest_month_by_region_revised_by_ERA5.nc",
    "IFS_proxy_statistics_hottest_regional_1981_2024_land.nc",
    region_grid_ds
)

grid_dp3 = load_and_test_trend(
    "Berkeley_hottest_month_by_region_revised_by_ERA5.nc",
    "IFS_proxy_trends_hottest_1981_2024.nc",
    "/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/region_mask_1deg.nc"
)

# Build comparison map
grid_comparison = xr.full_like(grid_ifs, np.nan, dtype=float)
for i in range(grid_ifs.shape[0]):
    for j in range(grid_ifs.shape[1]):
        val1 = grid_ifs[i, j].item()
        val2 = grid_dp3[i, j].item()
        if np.isnan(val1) or np.isnan(val2):
            continue
        if val1 == 1 and val2 == 1:
            grid_comparison[i, j] = 3
        elif val1 == 1:
            grid_comparison[i, j] = 1
        elif val2 == 1:
            grid_comparison[i, j] = 2
        else:
            grid_comparison[i, j] = 0

# Colormaps and normalization
cmap_ifs = mcolors.ListedColormap(["#D3D3D3", "#d580ff"])
cmap_dp3 = mcolors.ListedColormap(["#D3D3D3", "#ffa500"])
cmap_comparison = mcolors.ListedColormap(["#D3D3D3", "#d580ff", "#ffa500", "#4cbb17"])

norm_ifs = mcolors.BoundaryNorm([-0.1, 0.75, 1.1], cmap_ifs.N)
norm_dp3 = mcolors.BoundaryNorm([-0.1, 0.75, 1.1], cmap_dp3.N)
norm_comp = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap_comparison.N)

# Prepare plots
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(10, 14),
                        subplot_kw={'projection': ccrs.PlateCarree()})

titles = ["(a) IFS Statistics Fidelity hottest month 1981–2024",
          "(b) IFS Trends Fidelity hottest month 1981–2024",
          "(c) Fidelity Agreement Map"]

datasets = [grid_ifs, grid_dp3, grid_comparison]
cmaps = [cmap_ifs, cmap_dp3, cmap_comparison]
norms = [norm_ifs, norm_dp3, norm_comp]

for ax, data, title, cmap, norm in zip(axs, datasets, titles, cmaps, norms):
    ax.set_title(title, fontsize=11)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=":")
    im = ax.pcolormesh(region_grid_ds.longitude, region_grid_ds.latitude, data,
                       shading="auto", cmap=cmap, norm=norm, transform=ccrs.PlateCarree())

fig.tight_layout(pad=3.0)
fig.subplots_adjust(bottom=0.1)
cbar_ax = fig.add_axes([0.2, 0.06, 0.6, 0.008])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', ticks=[0, 1, 2, 3])
cbar.ax.set_xticklabels(["Fail", "Statistics Pass", "Trends Pass", "Both Pass"])
cbar.ax.tick_params(which='both', length=0)

plt.savefig("IFS_Fidelity_Comparison_hottestmonth_1981_2024_Berkeley.png", dpi=300)
plt.show()
